# Reverse Model: Fit Physics Parameters from Experimental Data

This notebook takes measured **(flywheel speed, ground landing distance)** data from a
topspin flywheel shooter, fits the physics model parameters, and then uses the fitted
model to predict **how long it takes for the ball's center to reach a given target height**
(e.g. hub center).

At the end, it fits a **closed-form equation** for `time = f(distance)` and `time = f(RPM)`
so the robot can compute time-of-flight with a simple formula — no simulation needed.

### What's fitted
- **N** – efficiency / velocity scaling factor
- **Cd** – drag coefficient (constrained near 0.47)

### What's fixed
- **Cl** = `0.2 * R_FUEL * W0 / V0` (standard topspin formula)

### How to use
1. Enter your measured data in the **Data Input** cell.
2. Set `TARGET_HEIGHT` to the hub center height you care about.
3. Run all cells.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, curve_fit

---
## ✏️ Data Input & Configuration

In [ ]:
# ──────────────────────────────────────────────────────────
# Raw experimental data
#   speed_rad_s : flywheel angular velocity (rad/s)
#   xA_inches   : distance from hub center to landing (in)
#
# Each speed has 12 trials.
# ──────────────────────────────────────────────────────────
raw_data = {
    # speed_rad_s : [xA values in inches for each trial]
    30: [106.8, 88.6, 97.2, 92.9, 97.2, 87.5, 108.9, 97.2, 101.5, 108.9, 99.3, 92.9],
    35: [127.6, 112.0, 122.4, 122.4, 130.7, 132.8, 147.2, 146.2, 143.1, 132.8, 135.9, 139.0],
    40: [143.1, 155.4, 172.7, 168.7, 161.5, 180.9, 181.9, 158.5, 192.0, 187.0, 191.0, 183.9],
    45: [203.2, 217.4, 206.2, 157.4, 204.2, 162.5, 236.5, 247.6, 200.2, 244.6, 237.6, 242.6],
    50: [236.5, 200.2, 247.6, 222.4, 195.1, 246.6, 286.9, 278.9, 292.0, 261.8, 294.0, 275.9],
    55: [235.5, 218.4, 265.8, 206.2, 193.1, 177.8, 292.0, 218.4, 225.4, 339.2, 290.9, 329.1],
}

# ── Convert to metric & compute averages ──
IN_TO_M = 0.0254  # inches → metres

speeds_rad  = []  # rad/s  (one per group)
speeds_rpm  = []  # RPM    (one per group)
avg_dist_m  = []  # avg landing distance in metres

# Also collect every individual trial for plotting
all_speeds_rpm = []
all_dists_m    = []

print(f"{'Speed':>8} {'RPM':>8} {'Trials':>6}  {'Avg xA (in)':>12} {'Avg xA (m)':>12}")
print('-' * 52)
for w, trials in sorted(raw_data.items()):
    rpm = w * 60 / (2 * math.pi)
    avg_in = np.mean(trials)
    avg_m  = avg_in * IN_TO_M
    speeds_rad.append(w)
    speeds_rpm.append(rpm)
    avg_dist_m.append(avg_m)
    for xa in trials:
        all_speeds_rpm.append(rpm)
        all_dists_m.append(xa * IN_TO_M)
    print(f"{w:>5} r/s {rpm:>8.1f} {len(trials):>6}  {avg_in:>12.1f} {avg_m:>12.4f}")

speeds_rpm  = np.array(speeds_rpm)
avg_dist_m  = np.array(avg_dist_m)
all_speeds_rpm = np.array(all_speeds_rpm)
all_dists_m    = np.array(all_dists_m)

In [ ]:
# ──────────────────────────────────────────────
# Target height (meters)  ← CHANGE THIS EASILY
# This is the HEIGHT OF THE HUB CENTER.
# The notebook computes when the ball's center
# reaches this height on descent.
# ──────────────────────────────────────────────
TARGET_HEIGHT = 1.872  # metres (hub center height)

print(f"Target height (hub center): {TARGET_HEIGHT} m")

In [ ]:
# ──────────────────────────────────────────────
# Physical constants
# ──────────────────────────────────────────────
R_FUEL  = 0.15 / 2        # ball radius (m)
D_WHEEL = 0.1016          # flywheel diameter (m)
MASS    = 0.20366297      # ball mass (kg)
G       = 9.81            # gravity (m/s²)
RHO     = 1.225           # air density (kg/m³)
A_CROSS = math.pi * R_FUEL**2

CL_BASE = 0.2             # base lift coeff (Cl = CL_BASE * R * W0 / V0)

LAUNCH_ANGLE_DEG = 72
THETA = math.radians(LAUNCH_ANGLE_DEG)

Y0 = 0.2032              # launcher exit height (m)

# Simulation time step
DELTA_T  = 0.0001         # (s)  — 0.1 ms
MAX_TIME = 5.0            # (s)

print("Constants loaded.")

---
## Forward Simulation Engine

Two simulation modes:
- **`simulate_to_ground`** – used for **fitting**: runs until the ball lands (`y ≤ 0`).
- **`simulate_to_height`** – used for **time lookup**: runs until the ball's **center** descends through `target_height` (hub center).

In [ ]:
def _compute_v0(rpm, N):
    """Exit velocity from RPM and efficiency."""
    return ((rpm * 2 * math.pi) / 60 * D_WHEEL / 2) / 2 * N


def simulate_to_ground(rpm, N, Cd):
    """
    Simulate until the ball hits the ground (y <= 0).
    Returns (x_distance, time) or (None, None).
    """
    V0 = _compute_v0(rpm, N)
    if V0 <= 0:
        return None, None

    W0 = V0 / R_FUEL
    Cl = CL_BASE * R_FUEL * W0 / V0
    k  = -0.5 * RHO * A_CROSS / MASS

    vx, vy = V0 * math.cos(THETA), V0 * math.sin(THETA)
    cx, cy = 0.0, Y0
    t = 0.0

    while t < MAX_TIME:
        speed = math.sqrt(vx*vx + vy*vy)
        ax = k * speed * (Cd * vx + Cl * vy)
        ay = -G + k * speed * (Cd * vy - Cl * vx)
        vx += ax * DELTA_T
        vy += ay * DELTA_T
        cx += vx * DELTA_T
        cy += vy * DELTA_T
        t  += DELTA_T
        if cy <= 0:
            return cx, t

    return None, None


def simulate_to_height(rpm, N, Cd, target_height):
    """
    Simulate until the ball's CENTER descends through target_height
    (i.e. the hub center height).
    Returns (x_distance, time) or (None, None).
    """
    V0 = _compute_v0(rpm, N)
    if V0 <= 0:
        return None, None

    W0 = V0 / R_FUEL
    Cl = CL_BASE * R_FUEL * W0 / V0
    k  = -0.5 * RHO * A_CROSS / MASS

    vx, vy = V0 * math.cos(THETA), V0 * math.sin(THETA)
    cx, cy = 0.0, Y0
    t = 0.0

    while t < MAX_TIME:
        speed = math.sqrt(vx*vx + vy*vy)
        ax = k * speed * (Cd * vx + Cl * vy)
        ay = -G + k * speed * (Cd * vy - Cl * vx)
        vx += ax * DELTA_T
        vy += ay * DELTA_T
        cx += vx * DELTA_T
        cy += vy * DELTA_T
        t  += DELTA_T
        # Ball center descending through hub center height
        if cy > target_height and vy < 0:
            return cx, t

    return None, None

print("Simulation functions ready.")

---
## Parameter Fitting

Fit **N** and **Cd** by minimising the sum-of-squared-errors between
simulated ground-landing distances and the averaged measurements.

In [ ]:
def objective(params):
    N, Cd = params
    sse = 0.0
    for rpm, d_meas in zip(speeds_rpm, avg_dist_m):
        d_sim, _ = simulate_to_ground(rpm, N, Cd)
        if d_sim is None:
            sse += 1e6
        else:
            sse += (d_sim - d_meas) ** 2
    return sse


# Initial guess
x0 = [0.92, 0.47]

# Bounds – Cd stays near 0.47, N is free
bounds = [
    (0.1, 50.0),    # N  (may be large due to gearing)
    (0.35, 0.60),   # Cd (near 0.47)
]

print("Fitting parameters (this may take a minute) …")
result = minimize(objective, x0, method='Nelder-Mead', bounds=bounds,
                  options={'maxiter': 5000, 'xatol': 1e-6, 'fatol': 1e-10})

N_fit, Cd_fit = result.x

print(f"\n✅  Optimisation finished  (success={result.success})")
print(f"   N     = {N_fit:.6f}")
print(f"   Cd    = {Cd_fit:.6f}")
print(f"   Cl    = {CL_BASE} · R·ω/v  (fixed)")
print(f"   SSE   = {result.fun:.10f}")

---
## Model vs. Data Comparison

In [ ]:
# Simulate a smooth curve across the full RPM range
rpm_lo, rpm_hi = speeds_rpm.min() * 0.9, speeds_rpm.max() * 1.1
rpm_curve = np.linspace(rpm_lo, rpm_hi, 200)
dist_curve = []
for r in rpm_curve:
    d, _ = simulate_to_ground(r, N_fit, Cd_fit)
    dist_curve.append(d)

# Per-group predictions
dist_pred = []
for r in speeds_rpm:
    d, _ = simulate_to_ground(r, N_fit, Cd_fit)
    dist_pred.append(d)

plt.figure(figsize=(10, 6))
# Individual trials (translucent)
plt.scatter(all_speeds_rpm, all_dists_m, color='gray', alpha=0.35, s=25, label='Individual trials')
# Averaged data
plt.scatter(speeds_rpm, avg_dist_m, color='black', s=70, zorder=5, label='Averaged data')
# Fitted model curve
plt.plot(rpm_curve, dist_curve, 'r-', linewidth=2, label='Fitted model')
plt.xlabel('RPM', fontsize=13)
plt.ylabel('Ground landing distance (m)', fontsize=13)
plt.title(f'Model Fit  –  N={N_fit:.4f}, Cd={Cd_fit:.4f}', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Residual table
print(f"\n{'rad/s':>6} {'RPM':>8}  {'Measured':>10}  {'Predicted':>10}  {'Error (m)':>10}")
print('-' * 50)
for w, rpm, dm, dp in zip(sorted(raw_data.keys()), speeds_rpm, avg_dist_m, dist_pred):
    err = (dp - dm) if dp is not None else float('nan')
    dp_str = f'{dp:.4f}' if dp is not None else 'N/A'
    print(f"{w:>6} {rpm:>8.1f}  {dm:>10.4f}  {dp_str:>10}  {err:>10.4f}")

---
## ⏱ Time to Hub Center (Simulation)

Using the fitted parameters, compute how long the ball's **center** takes
to descend through `TARGET_HEIGHT` (hub center) for a dense range of RPMs.
This data feeds the regression in the next section.

In [ ]:
# Generate a dense set of (RPM, distance, time) points
sim_rpms  = []
sim_dists = []
sim_times = []

for rpm in np.linspace(speeds_rpm.min(), speeds_rpm.max(), 300):
    d, t = simulate_to_height(rpm, N_fit, Cd_fit, TARGET_HEIGHT)
    if d is not None and t is not None:
        sim_rpms.append(rpm)
        sim_dists.append(d)
        sim_times.append(t)

sim_rpms  = np.array(sim_rpms)
sim_dists = np.array(sim_dists)
sim_times = np.array(sim_times)

# Print the per-speed-group values
print(f"Target height (hub center): {TARGET_HEIGHT} m\n")
print(f"{'rad/s':>6} {'RPM':>8}  {'Distance (m)':>13}  {'Time (s)':>10}  {'Time (ms)':>10}")
print('-' * 55)
for w, rpm in zip(sorted(raw_data.keys()), speeds_rpm):
    d, t = simulate_to_height(rpm, N_fit, Cd_fit, TARGET_HEIGHT)
    if d is not None:
        print(f"{w:>6} {rpm:>8.1f}  {d:>13.4f}  {t:>10.6f}  {t*1000:>10.2f}")
    else:
        print(f"{w:>6} {rpm:>8.1f}  {'—':>13}  {'—':>10}  {'—':>10}")

print(f"\nGenerated {len(sim_rpms)} simulation points for regression.")

---
## 📐 Time-of-Flight Regression (for the robot)

Fit simple closed-form equations so the robot can compute time-of-flight
**without running the full simulation**.

Two regressions:
1. `time = f(distance)` – if the robot knows the distance to hub
2. `time = f(RPM)` – if the robot knows the flywheel speed

In [ ]:
# ═══════════════════════════════════════════════
# Regression 1:  time = f(distance)
# ═══════════════════════════════════════════════

# Try polynomial fits of different degrees
best_deg_d = None
best_r2_d  = -1
for deg in [1, 2, 3]:
    coeffs = np.polyfit(sim_dists, sim_times, deg)
    pred   = np.polyval(coeffs, sim_dists)
    ss_res = np.sum((sim_times - pred)**2)
    ss_tot = np.sum((sim_times - np.mean(sim_times))**2)
    r2 = 1 - ss_res / ss_tot
    print(f"  degree {deg}:  R² = {r2:.10f}")
    if r2 > best_r2_d:
        best_r2_d  = r2
        best_deg_d = deg

# Fit the best-degree polynomial
coeffs_dist = np.polyfit(sim_dists, sim_times, best_deg_d)

print(f"\n✅  Best fit: degree {best_deg_d}  (R² = {best_r2_d:.10f})")
print(f"\n--- Copy this into the robot code ---")
terms = []
for i, c in enumerate(coeffs_dist):
    power = best_deg_d - i
    if power == 0:
        terms.append(f"{c:+.10e}")
    elif power == 1:
        terms.append(f"{c:+.10e} * d")
    else:
        terms.append(f"{c:+.10e} * d^{power}")
print(f"time(d) = {' '.join(terms)}")
print(f"\nCoefficients (highest power first): {list(coeffs_dist)}")

# Plot
d_range = np.linspace(sim_dists.min(), sim_dists.max(), 200)
plt.figure(figsize=(10, 5))
plt.plot(sim_dists, sim_times * 1000, 'b.', alpha=0.3, markersize=4, label='Simulation')
plt.plot(d_range, np.polyval(coeffs_dist, d_range) * 1000, 'r-', linewidth=2,
         label=f'Poly deg {best_deg_d} (R²={best_r2_d:.6f})')
plt.xlabel('Distance to hub (m)', fontsize=13)
plt.ylabel('Time to hub center (ms)', fontsize=13)
plt.title('time = f(distance)  —  for robot', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════
# Regression 2:  time = f(RPM)
# ═══════════════════════════════════════════════

best_deg_r = None
best_r2_r  = -1
for deg in [1, 2, 3]:
    coeffs = np.polyfit(sim_rpms, sim_times, deg)
    pred   = np.polyval(coeffs, sim_rpms)
    ss_res = np.sum((sim_times - pred)**2)
    ss_tot = np.sum((sim_times - np.mean(sim_times))**2)
    r2 = 1 - ss_res / ss_tot
    print(f"  degree {deg}:  R² = {r2:.10f}")
    if r2 > best_r2_r:
        best_r2_r  = r2
        best_deg_r = deg

coeffs_rpm = np.polyfit(sim_rpms, sim_times, best_deg_r)

print(f"\n✅  Best fit: degree {best_deg_r}  (R² = {best_r2_r:.10f})")
print(f"\n--- Copy this into the robot code ---")
terms = []
for i, c in enumerate(coeffs_rpm):
    power = best_deg_r - i
    if power == 0:
        terms.append(f"{c:+.10e}")
    elif power == 1:
        terms.append(f"{c:+.10e} * rpm")
    else:
        terms.append(f"{c:+.10e} * rpm^{power}")
print(f"time(rpm) = {' '.join(terms)}")
print(f"\nCoefficients (highest power first): {list(coeffs_rpm)}")

# Plot
rpm_range = np.linspace(sim_rpms.min(), sim_rpms.max(), 200)
plt.figure(figsize=(10, 5))
plt.plot(sim_rpms, sim_times * 1000, 'b.', alpha=0.3, markersize=4, label='Simulation')
plt.plot(rpm_range, np.polyval(coeffs_rpm, rpm_range) * 1000, 'r-', linewidth=2,
         label=f'Poly deg {best_deg_r} (R²={best_r2_r:.6f})')
plt.xlabel('RPM', fontsize=13)
plt.ylabel('Time to hub center (ms)', fontsize=13)
plt.title('time = f(RPM)  —  for robot', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔍 Quick Single-Speed Query

In [ ]:
QUERY_SPEED_RAD = 45  # ← change me (rad/s)

query_rpm = QUERY_SPEED_RAD * 60 / (2 * math.pi)
d, t = simulate_to_height(query_rpm, N_fit, Cd_fit, TARGET_HEIGHT)
if d is not None:
    # Compare simulation vs regression
    t_reg_d = np.polyval(coeffs_dist, d)
    t_reg_r = np.polyval(coeffs_rpm, query_rpm)
    print(f"At {QUERY_SPEED_RAD} rad/s ({query_rpm:.1f} RPM):")
    print(f"  Ball center reaches {TARGET_HEIGHT} m (hub center)")
    print(f"")
    print(f"  Simulation : {t:.6f} s  ({t*1000:.2f} ms)")
    print(f"  Regression (from dist  {d:.4f} m) : {t_reg_d:.6f} s  ({t_reg_d*1000:.2f} ms)")
    print(f"  Regression (from RPM  {query_rpm:.1f})  : {t_reg_r:.6f} s  ({t_reg_r*1000:.2f} ms)")
else:
    print(f"At {QUERY_SPEED_RAD} rad/s the ball never reaches {TARGET_HEIGHT} m.")